# 03 - Answer quality: does better retrieval produce better answers?

Notebook 02 established a 17-point nDCG gap between BM25 and the dense
retriever. This notebook asks the question that actually matters and that
most RAG writeups skip: **how much of that gap survives into the answers?**

Four conditions, 150 test questions, one variable changed at a time:

| Condition | Passages given to the generator |
|---|---|
| `closed_book` | none -- what the model already knows |
| `rag_bm25` | top-5 from BM25 (nDCG@10 = 0.237) |
| `rag_dense` | top-5 from bge-base (nDCG@10 = 0.406) |
| `oracle` | the human-judged relevant documents |

`closed_book` is the floor RAG has to beat. `oracle` is the ceiling, and it
is the one that separates a *retrieval* failure from a *generation*
failure.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
ROOT = Path.cwd().parent
report = json.loads((ROOT / "output/reports/generation_metrics.json").read_text())
records = json.loads((ROOT / "data/processed/generation/answers.json").read_text())
judgments = json.loads((ROOT / "data/processed/generation/judgments.json").read_text())

CONDITIONS = ["closed_book", "rag_bm25", "rag_dense", "oracle"]
LABELS = {"closed_book": "No retrieval", "rag_bm25": "RAG (BM25)",
          "rag_dense": "RAG (bge-base)", "oracle": "Oracle (gold)"}
print(f"{report['n_questions']} questions | generator {report['generator_model']} "
      f"| judge {report['judge_model']}")
print(f"evaluation cost: {report['estimated_cost_usd']}")

## 1. Start with the metrics that need no judge

Three of these come from a parser and the human relevance judgments. No
model opinion is involved, so they are the numbers to trust first.

In [ ]:
rows = []
for c in CONDITIONS:
    m = report["conditions"][c]
    rows.append({
        "condition": LABELS[c],
        "abstained": m.get("abstained"),
        "citation precision": m.get("citation_precision"),
        "invalid citations": m.get("invalid_citations"),
    })
pd.DataFrame(rows).set_index("condition")

**Abstention tracks retrieval quality.** The system declines to answer far
more often when handed BM25's weaker passages than the dense retriever's.
That is the property that separates a system that knows what it does not
know from one that confabulates fluently -- and the failure mode of a bad
RAG system is not silence, it is a confident answer wearing citations.

**Citation precision** -- the share of cited passages that a human
annotator actually marked relevant -- moves the same way. The oracle
condition scoring exactly 1.000 is a pipeline check rather than a finding:
by construction every oracle passage is gold, so anything below 1.0 would
have meant a bug in the context builder or the citation parser.

**Zero invalid citations** across 600 answers. The model never cited a
passage number that was not in front of it.

## 2. The judged metrics

In [ ]:
rows = []
for c in CONDITIONS:
    m = report["conditions"][c]
    rows.append({
        "condition": LABELS[c],
        "correct": m.get("correct"),
        "correct or partial": m.get("correct_or_partial"),
        "declined (judge)": m.get("declined"),
        "grounded": m.get("grounded"),
        "n": m.get("n_scored"),
    })
judged = pd.DataFrame(rows).set_index("condition")
judged

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(CONDITIONS))
w = 0.38
ax.bar(x - w/2, [report["conditions"][c].get("correct", 0) for c in CONDITIONS],
       w, label="correct", color="#4C72B0")
ax.bar(x + w/2, [report["conditions"][c].get("correct_or_partial", 0) for c in CONDITIONS],
       w, label="correct or partial", color="#9BB7D9")
ax.set_xticks(x); ax.set_xticklabels([LABELS[c] for c in CONDITIONS], rotation=15)
ax.set_ylabel("share of questions"); ax.set_title("Answer correctness by condition")
ax.legend(); plt.tight_layout(); plt.show()

## 3. Can the judge be trusted?

This is the part an LLM-judged evaluation cannot skip. The generator and
the primary judge are both `claude-opus-5`, which makes self-preference a
live risk, and a judge that contradicts itself cannot resolve a small gap
between conditions.

Two checks, neither of which is a substitute for human labels:

- **Self-consistency** -- the same judge re-runs a subset. Disagreement
  here is an upper bound on how finely it can discriminate.
- **Cross-model** -- a different model re-judges the same items. If the
  ranking survives, it is probably about the answers; if it does not, it is
  about the judge.

In [ ]:
cal = report["judge_calibration"]
print(f"self-consistency : {cal['self_consistency']}")
print(f"vs {cal['cross_model_id']:20s}: {cal['cross_model']}")

In [ ]:
# Where do the two judges disagree? A confusion matrix says whether the
# disagreement is noise around a boundary or a systematic difference in
# strictness -- the second would undermine the ranking, the first would not.
primary, cross = judgments["judge_correctness"], judgments["judge_correctness_cross"]
pairs = [(primary[k]["verdict"], cross[k]["verdict"])
         for k in cross if primary.get(k) and cross.get(k)]
order = ["correct", "partially_correct", "incorrect", "no_answer"]
cm = pd.crosstab(pd.Series([a for a, _ in pairs], name=f"{report['judge_model']}"),
                 pd.Series([b for _, b in pairs], name=cal["cross_model_id"])
                 ).reindex(index=order, columns=order, fill_value=0)
plt.figure(figsize=(6.5, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Verdict agreement between the two judges")
plt.tight_layout(); plt.show()
cm

## 4. Where the answers actually fail

Aggregates hide the shape of the failures. The interesting cases are the
ones where retrieval succeeded and the answer still went wrong, and the
ones where the system abstained but the passages were fine.

In [ ]:
from src.data.loader import load_split
corpus, queries, qrels = load_split("test")

rows = []
for key, rec in records.items():
    if rec["condition"] != "rag_dense" or not rec["answer"]:
        continue
    ans, qid = rec["answer"], rec["qid"]
    hits = sum(1 for d in rec["context_ids"] if d in qrels[qid])
    v = judgments["judge_correctness"].get(key, {}).get("verdict")
    rows.append({"qid": qid, "gold in context": hits,
                 "abstained": not ans["context_sufficient"], "verdict": v})
df = pd.DataFrame(rows)

retrieval_hit = df[df["gold in context"] > 0]
retrieval_miss = df[df["gold in context"] == 0]
print(f"retrieval found >=1 gold passage : {len(retrieval_hit):3d} questions")
print(f"  abstained anyway               : {retrieval_hit['abstained'].mean():.1%}")
print(f"  judged correct                 : {(retrieval_hit['verdict'] == 'correct').mean():.1%}")
print(f"\nretrieval found none            : {len(retrieval_miss):3d} questions")
print(f"  abstained (correct behaviour)  : {retrieval_miss['abstained'].mean():.1%}")
print(f"  judged correct                 : {(retrieval_miss['verdict'] == 'correct').mean():.1%}")

The split between these two groups is the useful diagnostic. Questions
where retrieval missed entirely are the ceiling on what any generator can
do; questions where it hit and the answer was still wrong are the ones a
better prompt or a better generator could fix.

## 5. Reading the abstentions honestly

The oracle condition still abstains on a meaningful share of questions
*even when handed the documents a human marked relevant*. That is not a
system failure -- it is a property of FiQA. A document can be topically
relevant without containing the answer, and it bounds every number above.

In [ ]:
ab = {LABELS[c]: report["conditions"][c].get("abstained")
      for c in CONDITIONS if "abstained" in report["conditions"][c]}
plt.figure(figsize=(7, 4))
plt.bar(list(ab), list(ab.values()), color=["#C44E52", "#DD8452", "#55A868"])
plt.ylabel("share of questions"); plt.title("Abstention rate: 'the passages don't support an answer'")
plt.tight_layout(); plt.show()
ab